## PROYECTO: PORTAL DE TICKETS PQRS
### ETAPA: CARGA (CAPA GOLD)
**OBJETIVO:** Generar agregaciones y KPIs listos para consumo analítico en herramientas de BI (ej. Power BI).

In [0]:
from pyspark.sql.functions import col, count, sum, when, current_timestamp

# Limpiamos widgets previos para evitar conflictos
dbutils.widgets.removeAll()

In [0]:
# 1. PARÁMETROS DINÁMICOS (WIDGETS)

dbutils.widgets.text("catalogo", "catalogo_pqrs")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "gold")

catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

print("✅ Parámetros cargados.")

In [0]:
# 2. LECTURA DE TABLA DESDE SILVER

df_silver = spark.table(f"{catalogo}.{esquema_source}.tickets_enriquecidos")

In [0]:
# 3. CREACIÓN DE DATAMART (AGREGACIONES Y KPIs PARA POWER BI)
from pyspark.sql.functions import col, count, sum, avg, to_date, hour, current_timestamp

spark.conf.set("spark.sql.ansi.enabled", "false")

# 📊 KPI 1: Rendimiento Extendido de Agentes
df_kpi_agentes = df_silver.groupBy("agente_asignado", "region_agente") \
    .agg(
        count("ticket_id").alias("total_tickets_atendidos"),
        sum("criticidad").alias("total_tickets_criticos"),
        avg("tiempo_espera_segundos").alias("tiempo_promedio_espera_seg")
    ).withColumn("fecha_actualizacion_kpi", current_timestamp())

# 📊 KPI 2: Análisis por Canales y Tecnología
df_kpi_tecnologia = df_silver.groupBy("canal_origen", "sistema_operativo") \
    .agg(
        count("ticket_id").alias("volumen_tickets"),
        sum("criticidad").alias("impacto_critico")
    ).withColumn("fecha_actualizacion_kpi", current_timestamp())

# 📊 KPI 3: Tendencia Operativa por Hora
df_kpi_tendencia = df_silver.withColumn("fecha_apertura", to_date(col("timestamp_apertura"))) \
    .withColumn("hora_apertura", hour(col("timestamp_apertura"))) \
    .groupBy("fecha_apertura", "hora_apertura") \
    .agg(
        count("ticket_id").alias("tickets_recibidos"),
        avg("tiempo_espera_segundos").alias("espera_promedio_por_hora")
    ).withColumn("fecha_actualizacion_kpi", current_timestamp())

In [0]:
display(df_kpi_agentes)
display(df_kpi_tecnologia)
display(df_kpi_tendencia)

In [0]:
# 4. GUARDADO EN CAPA GOLD (MÚLTIPLES TABLAS PARA POWER BI)

# Guardar KPI Agentes
df_kpi_agentes.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{esquema_sink}.kpi_agentes")

# Guardar KPI Tecnología
df_kpi_tecnologia.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{esquema_sink}.kpi_tecnologia")

# Guardar KPI Tendencias
df_kpi_tendencia.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{esquema_sink}.kpi_tendencias")

print(f"✅ Carga completada. Múltiples DataMarts guardados en el esquema: {esquema_sink}")